# P3 walkthrough: the learning agent

Run the cells top to bottom. Each part explains one file in `agent/`, lets you poke it, and ends with a
**Your turn** cell. Cells marked `TODO` are the ones you finish. Nothing here changes the repo unless a cell says so.

**The big picture in four lines**
1. The leg and the ground are the *environment* (done). The network turns sensor readings into an action (P1 and P2 build it).
2. **You build the agent**: the program that runs episodes, hands experience to the network, updates it, and measures the result.
3. Today a plain agent learns badly: **26% of episodes survive**, while one fixed action survives about **92.5%**.
4. **Goal:** find agent settings that make the network learn, so it matches that survival *and* lowers toe force. Freeze those settings on **24 Oct**.

Time knob: `QUICK = True` keeps every demo to a minute or two. Real runs use 300k steps (about 5 minutes for Network A).

In [ ]:
%pip install pandas numpy matplotlib

In [3]:
import os, sys, json, subprocess, time, tempfile
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

# Work from the Network/ folder so `import agent`, `import common` work.
ROOT = Path.cwd()
while not (ROOT / "agent").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert (ROOT / "agent").exists(), "open this notebook from inside the Network/ folder"
os.chdir(ROOT); sys.path.insert(0, str(ROOT))

QUICK = True
DEMO_STEPS = 8192 if QUICK else 300_000
print("working in", ROOT)

working in /mnt/usb-Seagate_FireCuda_SSD_00000000NABG06FP-0:0-part1/Ubuntu_Files/ME5418_Machine-Learning/ME5418_Project/STANCE/Claude-Files/Network_Agent


## Part 1. The environment: what the agent talks to

`common/env_setup.make_env()` builds the leg-and-ground environment. Every step the agent gets **110 numbers**
(10 time frames x 11 sensors, oldest first) and must give back **3 numbers in [-1, 1]**: target angle, stiffness K, damping B.
An episode is one 100-step stance. It ends early if the leg falls.

In [21]:
from common.env_setup import make_env, SENSOR_NAMES, N_SENSORS, to_norm_action, to_physical

env = make_env("train")
obs, info = env.reset(seed=1000)
print("observation shape:", obs.shape, "  action space:", env.action_space)
frames = obs.reshape(-1, N_SENSORS)           # 10 rows (frames), 11 columns (sensors)
print(pd.DataFrame(frames[-1:], columns=SENSOR_NAMES).T.rename(columns={0: "newest frame"}))

observation shape: (110,)   action space: Box(-1.0, 1.0, (3,), float32)
                   newest frame
ankle_angle            0.009018
ankle_rate             1.018212
ankle_torque           0.000000
vertical_grf           0.000000
pylon_compression      0.000000
shank_tilt             0.027243
shank_tilt_rate        1.018212
vertical_accel        -9.124960
prev_theta_d           0.000000
prev_K                 0.000000
prev_B                 0.000000


In [25]:
# One step with the middle action. The info dictionary is where toe force and the failure reason live.
obs, r, terminated, truncated, info = env.step(np.zeros(3, dtype=np.float32))
print("reward", round(float(r), 3), "| terminated", terminated, "| truncated", truncated)
print("info keys:", sorted(info))

reward 1.0 | terminated False | truncated False
info keys: ['body_height', 'dent_mm', 'k0', 'leg_tilt_deg', 'mu', 'normal_force_bw', 'toe_force_bw']


**Day-1 check 1 and 2.** The evaluator needs the name of the toe-force key in `info`. If the next cell prints `toe: None`,
open `common/evaluate.py` and fill in `INFO_KEYS` with the real key names you see above (the only edit allowed in `common/` besides `env_setup`).

In [26]:
from common.evaluate import run_episodes, INFO_KEYS, REFERENCE, summarise, print_summary
from common.policies import FixedPolicy

fixed = lambda th, K, B: FixedPolicy(to_norm_action(env, th, K, B))
eps = run_episodes(fixed(*REFERENCE), make_env("train"), list(range(1000, 1005)))
print("INFO_KEYS pinned:", INFO_KEYS)
print("toe peak found:", [round(e["toe_peak"], 2) for e in eps])

INFO_KEYS pinned: {'toe': None, 'total': None, 'reason': None}
toe peak found: [1.54, 1.37, 1.59, 1.63, 1.32]


## Part 2. The bar to beat: the best fixed action

A *fixed* action never changes: angle -20 deg, K 500, B 10, every step. It needs no learning and survives about 92.5% of episodes on your frozen environment.
`summarise` turns episodes into the numbers the report uses. (The full 200-episode reference is `python -m common.evaluate --reference`; it also writes
`results/reference_train_1000-1199.csv`, which you commit.)

In [28]:
N = 30 if QUICK else 200
seeds = list(range(1000, 1000 + N))
s = summarise(run_episodes(fixed(*REFERENCE), make_env("train"), seeds))
print_summary("best fixed action", s)


== best fixed action (30 episodes) ==
  survival_pct       96.667
  survival_ci        [83.3, 99.4]
  falls_by_reason    forward:1
  mean_steps         99.933
  toe_peak_bw        1.561
  total_peak_bw      1.794
  mean_theta_deg     -20.000
  mean_K             500.000
  mean_B             10.000
  mean_return        96.470
  return_ci          [91.81, 101.13]
  ceiling_pct        83.333


**Your turn.** Try other fixed actions and see how survival and toe force trade off. Soft (low K) is gentle on the toe but is it safe?

In [32]:
for th, K, B in [(-20, 500, 10), (-10, 250, 5), (0, 100, 2), (5, 50, 1)]:
    s = summarise(run_episodes(fixed(th, K, B), make_env("train"), seeds))
    print(f"theta {th:>4}  K {K:>4}  B {B:>3}  ->  survival {s['survival_pct']:5.1f}%   toe peak {s['toe_peak_bw']:.2f} BW")

theta  -20  K  500  B  10  ->  survival  96.7%   toe peak 1.56 BW
theta  -10  K  250  B   5  ->  survival  66.7%   toe peak 1.29 BW
theta    0  K  100  B   2  ->  survival  10.0%   toe peak 0.96 BW
theta    5  K   50  B   1  ->  survival  10.0%   toe peak 0.88 BW


## Part 3. `agent/config.py`: the knobs

Every training setting lives in one dictionary, `AGENT_CONFIG`. `train.merged_config` stacks four layers, later ones win:
**agent defaults, the network's own `CONFIG`, `--smoke`, then your `--set` values.** It stops with an error on unknown keys or a key defined twice.
What the main knobs do:

| Key | Plain meaning |
|---|---|
| `gamma` | how far ahead the agent cares. Falls happen late (step 75-90), so a low `gamma` makes the fall penalty almost invisible |
| `learning_rate`, `n_epochs` | size and number of weight updates per batch of experience; too big means unstable |
| `ent_coef` | bonus for trying different things (exploration) |
| `norm_reward` | rescales rewards by their running size; may hide how large the fall penalty is |
| `residual`, `anchor`, `scale` | policy outputs a small offset from a good action instead of the whole action (Part 4) |

In [33]:
from agent.config import AGENT_CONFIG, SMOKE
from agent.train import load_network, merged_config

net = load_network("mlp")
cfg = merged_config(net, overrides=["gamma=0.995", "learning_rate=1e-4"])
print(pd.Series({k: cfg[k] for k in ["gamma", "learning_rate", "n_epochs", "ent_coef", "norm_reward", "residual", "total_steps"]}))
print("\nnetwork's own settings:", net.CONFIG)

gamma             0.995
learning_rate    0.0001
n_epochs             10
ent_coef            0.0
norm_reward        True
residual          False
total_steps      300000
dtype: object

network's own settings: {'frames': 10, 'drop_sensors': [], 'hidden': [256, 256], 'activation': 'tanh', 'layer_norm': False, 'head': [64], 'log_std_init': 0.0}


In [34]:
# The safety checks in action: both of these should print an error message, not train.
for bad in (["gamma_typo=1"], ["gamma"]):
    try:
        merged_config(net, overrides=bad)
    except SystemExit as e:
        print("stopped ->", e)

stopped -> --set: unknown key 'gamma_typo'. Known keys: ['activation', 'anchor', 'batch_size', 'clip_range', 'curve_episodes', 'curve_every', 'drop_sensors', 'ent_coef', 'eval_episodes', 'frames', 'gae_lambda', 'gamma', 'head', 'hidden', 'layer_norm', 'learning_rate', 'log_std_init', 'max_grad_norm', 'n_envs', 'n_epochs', 'n_steps', 'norm_obs', 'norm_reward', 'residual', 'scale', 'subproc', 'total_steps', 'vf_coef']
stopped -> --set expects key=value, got 'gamma'


## Part 4. `agent/actions.py`: the residual action, and the edge problem

Instead of asking the network for the whole action, ask for a small **offset** from a good action (the anchor):

`env_action = clip(anchor + scale * policy_action, -1, 1)`

An untrained network outputs about zero, so it starts out behaving like the anchor, i.e. at ~92% survival instead of ~11%.
**The catch:** the anchor (-20 deg, K 500, B 10) is at the *edge* of every range. Anything pointing outward is clipped away, so half of the network's output range does nothing.

In [35]:
from agent.actions import ResidualAction

anchor_n = to_norm_action(env, *REFERENCE)
print("anchor in normalised units [-1, 1]:", np.round(anchor_n, 3), " (+-1 means 'at the edge')")

scale = 0.3
a = np.linspace(-1, 1, 201)                       # every output the network could give
for d, name in enumerate(["angle", "K", "B"]):
    out = np.clip(anchor_n[d] + scale * a, -1, 1)
    dead = np.mean(np.abs(out) == 1)              # share of outputs squashed onto the edge
    print(f"{name:>5}: output range wasted by the clip = {dead:4.0%}")

anchor in normalised units [-1, 1]: [-1.  1.  1.]  (+-1 means 'at the edge')
angle: output range wasted by the clip =  50%
    K: output range wasted by the clip =  50%
    B: output range wasted by the clip =  50%


In [36]:
# The wrapper does exactly this on a real env. Zero output -> exactly the anchor.
renv = ResidualAction(make_env("train"), REFERENCE, (0.3, 0.3, 0.3))
print("zero policy output ->", np.round(to_physical(renv.env, renv.action(np.zeros(3))), 1), "(angle, K, B)")
print("max push outward   ->", np.round(to_physical(renv.env, renv.action(np.ones(3))), 1))
print("max push inward    ->", np.round(to_physical(renv.env, renv.action(-np.ones(3))), 1))

zero policy output -> [-20. 500.  10.] (angle, K, B)
max push outward   -> [-14. 500.  10.]
max push inward    -> [-20.  250.6   6.4]


### Your turn: remove the dead zone

The simplest fix is **move the anchor inside the range**, far enough that anchor +- scale always fits. Finish `inside_anchor`
(clip each normalised anchor value to the interval `[-1 + scale, 1 - scale]`). The check cell tells you if it is right.

In [45]:
def inside_anchor(anchor_n, scale):
    """TODO: return the anchor moved just far enough inward that anchor +- scale stays inside [-1, 1]."""
    return np.clip(anchor_n, -1 + scale, 1 - scale)

In [46]:
def check_inside_anchor(f):
    for s in (0.1, 0.3, 0.6):
        moved = np.asarray(f(anchor_n, s), dtype=float)
        assert np.all(moved + s <= 1 + 1e-6) and np.all(moved - s >= -1 - 1e-6), f"scale {s}: still hits the edge"
        centred = np.array([0.0, 0.2, -0.2])          # already inside: must stay put
        assert np.allclose(f(centred, 0.3), centred), "an anchor already inside the range must not move"
    print("correct")

check_inside_anchor(inside_anchor)

correct


In [47]:
# Solution: look only after trying.
def inside_anchor_solution(anchor_n, scale):
    return np.clip(anchor_n, -1 + scale, 1 - scale)
check_inside_anchor(inside_anchor_solution)

correct


**The price of moving the anchor:** it is no longer the best fixed action, so the starting survival drops. Measure it, then decide:
move the anchor, use a smaller `scale`, map only inward, or show the clip is harmless. Whatever you pick needs numbers in Report 2.

In [48]:
for scale in (0.1, 0.3):
    moved = inside_anchor_solution(anchor_n, scale)
    th, K, B = to_physical(env, moved)
    s = summarise(run_episodes(FixedPolicy(moved.astype(np.float32)), make_env("train"), seeds))
    print(f"scale {scale}: anchor becomes angle {th:6.1f}  K {K:5.0f}  B {B:4.1f}  -> start survival {s['survival_pct']:5.1f}%  toe {s['toe_peak_bw']:.2f} BW")
print("(config form: --set residual=True \"anchor=[%.1f,%.0f,%.1f]\")" % (th, K, B))

scale 0.1: anchor becomes angle  -18.0  K   397  B  8.6  -> start survival  96.7%  toe 1.51 BW
scale 0.3: anchor becomes angle  -14.0  K   251  B  6.4  -> start survival  86.7%  toe 1.37 BW
(config form: --set residual=True "anchor=[-14.0,251,6.4]")


## Part 5. `agent/train.py`: the main program, step by step

`train.main()` does these things in order. Here they are one at a time, so you can see what each returns.
`model.learn()` is where the lecture's loop runs inside Stable-Baselines3: reset, ask the network for actions, step, record,
and every `n_steps x n_envs` samples, update the weights.

In [49]:
from agent.train import make_wrappers, make_venv, build_model, print_architecture, count_params

cfg = merged_config(net, smoke=True)                 # tiny settings, fast
wrappers = make_wrappers(cfg, net)                   # network's input selection (+ residual if enabled)
venv = make_venv(cfg, wrappers, seed=0)              # parallel envs + running-average normalisation
model = build_model(cfg, net, venv, seed=0)          # PPO around the network
print_architecture(model, venv, net)

network: networkA_mlp
observation: (110,)   action: (3,)
ActorCriticPolicy(
  (features_extractor): HistoryMLP(
    (net): Sequential(
      (0): Linear(in_features=110, out_features=256, bias=True)
      (1): Tanh()
      (2): Linear(in_features=256, out_features=256, bias=True)
      (3): Tanh()
    )
  )
  (pi_features_extractor): HistoryMLP(
    (net): Sequential(
      (0): Linear(in_features=110, out_features=256, bias=True)
      (1): Tanh()
      (2): Linear(in_features=256, out_features=256, bias=True)
      (3): Tanh()
    )
  )
  (vf_features_extractor): HistoryMLP(
    (net): Sequential(
      (0): Linear(in_features=110, out_features=256, bias=True)
      (1): Tanh()
      (2): Linear(in_features=256, out_features=256, bias=True)
      (3): Tanh()
    )
  )
  (mlp_extractor): MlpExtractor(
    (policy_net): Sequential(
      (0): Linear(in_features=256, out_features=64, bias=True)
      (1): Tanh()
    )
    (value_net): Sequential(
      (0): Linear(in_features=256, out_f

**The loss PPO minimises** (computed inside the library, so you describe it in the report rather than code it, unless the professor says otherwise):

`loss = -(clipped policy objective) + vf_coef * (value error)^2 - ent_coef * entropy`

Gradients are clipped to length `max_grad_norm`, then Adam applies them. Next, train briefly with the learning-curve logger attached.

In [50]:
from agent.curve import CurveCallback

run_dir = Path(tempfile.mkdtemp(prefix="p3_demo_"))
cb = CurveCallback(dict(cfg, curve_every=2048, curve_episodes=3), wrappers, net.POLICY, run_dir / "curve.csv")
t0 = time.time()
model.learn(4096, callback=cb)
print(f"trained 4096 steps in {time.time()-t0:.0f}s")
pd.read_csv(run_dir / "curve.csv")

  curve: {'timesteps': 2048, 'minutes': 0.24, 'survival_pct': 0.0, 'mean_return': 60.17, 'toe_peak_bw': 1.094, 'mean_K': 50.8, 'mean_theta_deg': -0.08}
  curve: {'timesteps': 4096, 'minutes': 0.48, 'survival_pct': 0.0, 'mean_return': 59.11, 'toe_peak_bw': 1.115, 'mean_K': 57.5, 'mean_theta_deg': -1.4}
trained 4096 steps in 31s


,timesteps,minutes,survival_pct,mean_return,toe_peak_bw,mean_K,mean_theta_deg
0,2048,0.24,0.0,60.17,1.094,50.8,-0.08
1,4096,0.48,0.0,59.11,1.115,57.5,-1.40


`curve.csv` is the learning curve for the report. Watch **`mean_theta_deg`** as well as survival: the known failure is the angle drifting up towards +5 while survival stays low.

In [51]:
# Evaluate the freshly trained (4096-step, so poor) model with the shared protocol. The adapter makes it look like any other policy.
vn = venv; vn.training = False
s = summarise(run_episodes(net.POLICY(model, vn), make_env("train", wrappers), seeds[:10]))
print_summary("tiny smoke model", s)
venv.close()


== tiny smoke model (10 episodes) ==
  survival_pct       20.000
  survival_ci        [5.7, 51.0]
  falls_by_reason    forward:8
  mean_steps         85.800
  toe_peak_bw        0.856
  total_peak_bw      2.044
  mean_theta_deg     -0.215
  mean_K             51.428
  mean_B             2.333
  mean_return        64.897
  return_ci          [52.82, 76.98]
  ceiling_pct        20.000


## Part 6. Run real experiments

`run_exp` calls the same command you would type in a terminal (`python -m agent.train ...`) so the results land in `runs/` and `results/` exactly as in the report.
With `QUICK=True` it uses a few thousand steps just to prove the plumbing; set `QUICK=False` (or run the commands in a terminal) for the real 300k-step runs.
**One change per run, seed 0.**

In [52]:
def run_exp(name, steps=None, network="mlp", **overrides):
    """Train one run via the real CLI and return its learning curve as a DataFrame."""
    sets = [f"total_steps={steps or DEMO_STEPS}"] + [f"{k}={v!r}".replace("'", "") if not isinstance(v, str) else f"{k}={v}" for k, v in overrides.items()]
    sets = [x.replace(" ", "") for x in sets]
    env_ = {k: v for k, v in os.environ.items()}
    env_["PYTHONPATH"] = os.pathsep.join(p for p in env_.get("PYTHONPATH", "").split(os.pathsep) if p and "/opt/ros" not in p)
    cmd = [sys.executable, "-m", "agent.train", "--network", network, "--run-name", name, "--set", *sets]
    out = subprocess.run(cmd, capture_output=True, text=True, env=env_)
    if out.returncode:
        print(out.stdout[-1500:], out.stderr[-1500:]); raise RuntimeError("run failed")
    d = pd.read_csv(ROOT / "runs" / ("networkA_mlp" if network == "mlp" else "networkB_lstm") / f"{name}_s0" / "curve.csv")
    d["run"] = name
    return d

def plot_curves(curves):
    fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
    for d in curves:
        ax[0].plot(d.timesteps, d.survival_pct, marker="o", label=d.run[0])
        ax[1].plot(d.timesteps, d.mean_theta_deg, marker="o", label=d.run[0])
    ax[0].set(title="survival (%)", xlabel="steps"); ax[1].set(title="mean target angle (deg)", xlabel="steps")
    ax[0].legend(); ax[1].axhline(-20, ls=":", c="grey")
    plt.tight_layout(); plt.show()

base = run_exp("nb_base", curve_every=2048, curve_episodes=5, eval_episodes=10)
base

,timesteps,minutes,survival_pct,mean_return,toe_peak_bw,mean_K,mean_theta_deg,run
0,2048,0.14,40.0,76.60,1.073,49.9,0.03,nb_base
1,4096,0.30,40.0,76.52,1.089,50.0,-0.09,nb_base
2,6144,0.48,40.0,75.92,1.079,51.4,0.36,nb_base
3,8192,0.70,40.0,75.62,1.077,52.7,-0.17,nb_base


In [53]:
# Your turn: one change vs the baseline. Pick from the checklist below. Examples:
resid = run_exp("nb_resid", curve_every=2048, curve_episodes=5, eval_episodes=10, residual=True)
plot_curves([base, resid])

KeyboardInterrupt: 

### Experiment checklist (same as `P3_agent.md`)

Run each at 300k steps, seed 0, and write the final row of `results/networkA_mlp.csv` in your table. Judge by **survival, toe force and `mean_theta_deg`**, not return.
Differences under about 5 survival points are ties until you run a second seed.

| Run name | Change | Why |
|---|---|---|
| `base` | none | should land near 26% |
| `nonr` | `norm_reward=False` | normalisation may shrink the fall penalty |
| `g995` | `gamma=0.995` | makes the late fall count more |
| `lowlr` | `learning_rate=1e-4`, `n_epochs=5` | slower, steadier updates |
| `ent` | `ent_coef=0.01` | more exploration |
| `std` | `log_std_init=-1.0` | less starting noise |
| `long` | `total_steps=1000000` | slow learning or no learning? |
| `resid` | `residual=True` | start from the best fixed action |
| `resid_in` | `residual=True`, `anchor=[-15,300,6]` | fixes the edge problem |
| `resid_s1`, `resid_s6` | `scale=[0.1,0.1,0.1]`, `[0.6,0.6,0.6]` | how big may the offset be |

In [ ]:
# The whole table in one go. With QUICK=False this takes about 5 min per run, so start it and do something else.
EXPERIMENTS = {
    "nonr":     dict(norm_reward=False),
    "g995":     dict(gamma=0.995),
    "lowlr":    dict(learning_rate=1e-4, n_epochs=5),
    "ent":      dict(ent_coef=0.01),
    "std":      dict(log_std_init=-1.0),
}
RUN_ALL = False          # set True to run the dictionary above
if RUN_ALL:
    curves = [run_exp(n, curve_every=max(2048, DEMO_STEPS // 8), curve_episodes=20 if not QUICK else 5, **kw) for n, kw in EXPERIMENTS.items()]
    plot_curves([base] + curves)

In [ ]:
# Every finished run appends a row here. Read it as your results table.
rows = pd.read_csv(ROOT / "results" / "networkA_mlp.csv") if (ROOT / "results" / "networkA_mlp.csv").exists() else pd.DataFrame()
cols = ["run", "split", "survival_pct", "toe_peak_bw", "mean_theta_deg", "mean_K", "minus_reference"]
rows[[c for c in cols if c in rows.columns]] if len(rows) else "no rows yet"

## Part 7. Finish the job

Do these in the repo (not in this notebook). Tick them off as you go.

- [ ] **Today:** the three day-1 checks pass (Part 1, `--show-info`, `--reference` gives about 92.5% and 1.74 BW). Commit `INFO_KEYS` and `results/reference_train_1000-1199.csv`. Message the professor the three questions (SB3 allowed? hand-written loss? separate deadlines?).
- [ ] **Baseline and the learning problem:** the checklist table above. Write the numbers in a table as you go.
- [ ] **Residual action:** decide between moving the anchor, mapping only inward (edit `ResidualAction.action`), or keeping the clip with data. Add a test to `agent/test_agent.py` for whatever you change.
- [ ] **22 Oct:** write the best settings so far into `agent/config.py` (`AGENT_CONFIG`), commit, post "config v1".
- [ ] **24 Oct:** freeze it, tell Muthu and Mukul.
- [ ] **27-29 Oct:** Report 2 parts: the training loop, the settings and why, the PPO loss above, what you tried (including what failed), then assemble and check every number against `results/*.csv`.
- [ ] **31 Oct - 6 Nov:** fix `render()` in the environment (it imports a deleted module), record the rollout video with `VecVideoRecorder` (needs `moviepy`), run the final benchmark.

When you change `agent/config.py`, run `python -m pytest agent common -q` before committing.